# Contrastive Generation: the experiments of the post

This notebook reproduces every figure, animation and number of `contrastive_generation.md`. The functions it calls
live in `assess.py` next to it; training runs are launched as subprocesses of `assess.py`, several at a time.

* Run artifacts (checkpoints, logs, exported arrays) go to `demo/artifacts/`.
* Figures and animations of the post go to `materials/`, where the post links them.
* Everything runs on CPU. With 12 parallel processes the whole notebook takes about 40 minutes; finished runs are
  skipped on a re-run, so it can be interrupted and resumed.

| Part | Post | Produces |
|---|---|---|
| 0 | Sections 2 and 4 | numerical checks of the identities |
| 1 | Sections 1 to 3 | the flow-matching teacher |
| 2 | Section 2 | `trajectory_distillation.gif` |
| 3 | Section 4 | `f2_tangent_vs_chord.png`, the pos / neg code |
| 4 | Section 6 | the table, `samples_k1.gif`, `samples_k2.gif`, `directions.gif`, `pull_push_anchors.png`, the ablations |
| 5 | Section 8 | the table, `kl_vs_n_drift_drift_gauss_kneg.png`, `kernel_samples.gif`, `drift_directions.gif` |
| 6 | | a check that every asset the post links exists |

In [1]:
import inspect
import json
import os
import re
import sys
from pathlib import Path

import numpy as np
import torch
from IPython.display import Markdown, display

sys.path.insert(0, str(Path.cwd()))
import assess as A

PARALLEL = min(12, os.cpu_count())
PRESET, DEVICE, STEPS, SEEDS = "smoke", "cpu", 3000, (0, 1, 2)
FPS, HOLD = 8, 12
COMMON = ["--preset", PRESET, "--device", DEVICE, "--steps", str(STEPS), "--snap-count", "61", "--snap-n", "2000"]
A.MATERIALS.mkdir(exist_ok=True)


def show(*names):
    # Linked rather than embedded, so the notebook stays small; paths are relative to demo/.
    display(Markdown("\n".join(f"![{n}](../materials/{n})" for n in names)))


def jobs_for(variant, root, seeds=SEEDS, extra=(), common=COMMON):
    return [["--variant", variant, "--seed", str(s), "--root", str(root), *common, *extra] for s in seeds]


print("artifacts:", A.ART, "\nmaterials:", A.MATERIALS, "\nparallel processes:", PARALLEL)

artifacts: /Users/delcher/Desktop/self-info/contrastive-gen/demo/artifacts 
materials: /Users/delcher/Desktop/self-info/contrastive-gen/materials 
parallel processes: 12


## 0. The identities behind the post

**Master formula as a loss.** For a detached displacement $g = x_{\text{pos}} - x_{\text{neg}}$, the loss
$W\,\|\hat x - \mathrm{sg}(\hat x + g)\|^2$ has gradient $-2W(\partial\hat x/\partial\theta)^\top g$ (Section 2).
**Kernel posterior.** Under the empirical distribution of samples $y_j$, $\mathbb E[x_r\mid x_{t'}]$ is a
Gaussian-kernel mean of the $y_j$ and equals Tweedie's formula on the noised mixture (Section 8, Appendix G).
**Drift equilibrium.** The drift field vanishes when positives and negatives coincide (Section 8).

In [2]:
torch.manual_seed(0)
net = torch.nn.Linear(2, 2)
x_in, g, W = torch.randn(64, 2), torch.randn(64, 2), torch.rand(64, 1)
loss_grad = torch.autograd.grad(A.pull_push(net(x_in), g, W), net.weight)[0]
formula = torch.autograd.grad((-2 * W * g.detach() * net(x_in)).sum(-1).mean(), net.weight)[0]
print("master formula     max |difference| =", (loss_grad - formula).abs().max().item())

y = torch.randn(7, 2, dtype=torch.float64)
x = torch.randn(5, 2, dtype=torch.float64, requires_grad=True)
t, r = torch.full((5, 1), 0.7, dtype=torch.float64), torch.full((5, 1), 0.3, dtype=torch.float64)
a = (1 - t) / (1 - r)
var = t ** 2 - a ** 2 * r ** 2
logp = torch.logsumexp(-((x[:, None] - a[:, None] * y[None]) ** 2).sum(-1) / (2 * var), -1)
tweedie = (x + var * torch.autograd.grad(logp.sum(), x)[0]) / a
print("kernel posterior   max |difference| =",
      (A.kernel_posterior(x.detach(), y, t, r, self_mask=False) - tweedie).abs().max().item())

ys = torch.randn(300, 2)
V, _ = A.drift_field(torch.randn(300, 2), ys, ys, self_neg=False)
print("drift at p = q     max |V|          =", V.abs().max().item())

master formula     max |difference| = 1.4901161193847656e-08
kernel posterior   max |difference| = 2.7755575615628914e-16
drift at p = q     max |V|          = 0.0


## 1. The teacher

A 4×256 MLP trained by flow matching on two moons (2000 steps). Its weights are deterministic: the hash printed
below is the one recorded by every run of the post.

In [3]:
teacher_eval = A.run_teacher(PRESET, DEVICE)
print("teacher sha:", A.teacher_sha(), "(the post's runs: 5fb539f6e62b)")

[teacher] step      0 loss 3.8979 (0s)


[teacher] step    100 loss 3.0146 (0s)


[teacher] step    200 loss 3.0085 (1s)


[teacher] step    300 loss 2.9443 (1s)


[teacher] step    400 loss 2.8893 (1s)


...

[teacher] step   1800 loss 2.4953 (5s)


[teacher] step   1900 loss 2.5777 (5s)


[teacher] step   1999 loss 2.6661 (6s)


[teacher] SW2 rk4 0.0406  floor 0.0781  euler1 0.9237  accept=True


teacher sha: 5fb539f6e62b (the post's runs: 5fb539f6e62b)


## 2. Trajectory distillation (Section 2)

The student regresses its flow map onto the teacher's multi-step simulation: `sim-on-all`, three seeds.

In [4]:
DIST = A.ART / "distribution" / "runs"
A.run_jobs(jobs_for("sim-on-all", DIST), PARALLEL)
A.anim_single(DIST, "sim-on-all", "trajectory_distillation",
              "Trajectory distillation: regress the student's flow map onto the teacher's simulated flow map", PRESET, FPS, HOLD)
show("trajectory_distillation.gif")

[jobs] 0 already done, 3 to run


[sim-on-all s1] step   3000 sw2_k1 0.0650 sw2_k4 0.0403 coup 0.0253 | [sim-on-all s1] done in 73s


[sim-on-all s0] step   3000 sw2_k1 0.0642 sw2_k4 0.0396 coup 0.0248 | [sim-on-all s0] done in 73s


[sim-on-all s2] step   3000 sw2_k1 0.0631 sw2_k4 0.0405 coup 0.0246 | [sim-on-all s2] done in 73s


[anim] /Users/delcher/Desktop/self-info/contrastive-gen/materials/trajectory_distillation.gif / .mp4 (73 frames)


![trajectory_distillation.gif](../materials/trajectory_distillation.gif)

## 3. MeanFlow and DMD as pull / push (Section 4)

The schematic figure, then the positive / negative constructions and the one loss they share.

In [5]:
A.f2_tangent_vs_chord()
show("f2_tangent_vs_chord.png")
for fn in (A.pull_push, A.sim_pos_neg, A.mf_pos_neg, A.dmd_pos_neg, A.level_pred, A.renoise):
    print(inspect.getsource(fn))

[figures] /Users/delcher/Desktop/self-info/contrastive-gen/materials/f2_tangent_vs_chord.png


![f2_tangent_vs_chord.png](../materials/f2_tangent_vs_chord.png)

def pull_push(y, g, W=None):
    """mean W |y - sg(y + g)|^2: its gradient on y is -2 W g, so descent moves y along g.

    y is whatever the student's parameters produce: a sample x_pred with g = x_pos - x_neg, or a
    velocity u with g = u_pos - u_neg.
    """
    per = ((y - (y + g.detach()).detach()) ** 2).sum(-1, keepdim=True)
    return (per if W is None else W * per).mean()

@torch.no_grad()
def sim_pos_neg(teacher, u, x_t, t, r, n_rk4):
    """Trajectory consistency: positive is the teacher's simulated chord, negative is yourself."""
    return teacher.chord(x_t, t, r, n_rk4), u.detach()

@torch.no_grad()
def mf_pos_neg(student, u, x_t, t, r, v_target, tangent, method="jvp", eps_cd=1e-2):
    """MeanFlow: positive is the target's tangent, negative is the student's self-induced
    tangent u + J (computed from the student itself)."""
    J = mf_J(student, x_t, t, r, tangent, method, eps_cd)
    return v_target, u.detach() + J, J

@torch.no_grad()
def dmd_pos_neg(v_teacher, fake

## 4. Toy experiments (Section 6)

Six students (DMD and MeanFlow; on- and off-policy; $r = 0$ and random $r$), three seeds each, 3000 steps.
The export evaluates every seed's EMA student on held-out data (KL and SW2 at 1, 2 and 4 steps), collects the
sample clouds over training and the pos / neg construction on fixed anchors.

In [6]:
jobs = [j for run, _ in A.DEMO for j in jobs_for(run, DIST, extra=["--milestones"])]
A.run_jobs(jobs, PARALLEL)
demo_npz = A.export_distribution(DIST, PRESET, A.pick_device(DEVICE))

[jobs] 0 already done, 18 to run


[mf-off-r0 s2] step   3000 sw2_k1 0.0884 sw2_k4 0.1479 coup 0.0547 | [mf-off-r0 s2] done in 44s


[mf-off-r0 s1] step   3000 sw2_k1 0.0980 sw2_k4 0.1313 coup 0.0457 | [mf-off-r0 s1] done in 44s


[mf-off-r0 s0] step   3000 sw2_k1 0.0745 sw2_k4 0.1665 coup 0.0523 | [mf-off-r0 s0] done in 44s


[mf-on-r0 s2] step   3000 sw2_k1 0.0879 sw2_k4 0.0878 coup 0.0474 | [mf-on-r0 s2] done in 51s


...

[demo] mf-off-r0: 61 snapshots, 3 seeds


[demo] dmd-on-all: 61 snapshots, 3 seeds


[demo] mf-on-all: 61 snapshots, 3 seeds


[demo] /Users/delcher/Desktop/self-info/contrastive-gen/demo/artifacts/assets/distribution_demo.npz



Final test KL vs held-out data (nats, kNN k=5; mean ± sd over seeds). Data floor: KL(rev) -0.002, KL(fwd) -0.004, SW2 0.042
setup                                   K=1 rev          fwd       K=2 rev          fwd       K=4 rev          fwd
1  DMD, on-policy, r = 0            0.445±0.015  0.239±0.029   0.497±0.021  0.317±0.015   1.715±0.090  1.439±0.093
2  MeanFlow, on-policy, r = 0       1.123±0.141  1.080±0.135   1.682±0.505  1.380±0.307   0.849±0.127  0.723±0.121
3  DMD, off-policy, r = 0           1.069±0.149  0.854±0.207   0.510±0.023  0.311±0.013   0.621±0.017  0.355±0.014
4  MeanFlow, off-policy, r = 0      0.843±0.060  0.820±0.039   1.369±0.452  1.203±0.337   0.685±0.077  0.746±0.063
5  DMD, on-policy, r random         0.514±0.033  0.305±0.013   0.475±0.007  0.363±0.016   1.925±0.061  1.663±0.065
6  MeanFlow, on-policy, r random    0.953±0.371  0.794±0.361   0.761±0.056  0.689±0.038   0.670±0.048  0.381±0.050


The table of Section 6 (the post shows the first four numeric columns; the four-step KL is quoted in the text):

In [7]:
A.table_distribution(DIST)

| Setup | SW2, 1 step | Coupling, 1 step | KL(student||data), 1 step | KL, 2 steps | KL, 4 steps |
|---|---|---|---|---|---|
| 1  DMD, on-policy, r = 0 | 0.072 | 1.259 | 0.45 | 0.50 | 1.71 |
| 2  MeanFlow, on-policy, r = 0 | 0.099 | 0.060 | 1.12 | 1.68 | 0.85 |
| 3  DMD, off-policy, r = 0 | 0.246 | 2.520 | 1.07 | 0.51 | 0.62 |
| 4  MeanFlow, off-policy, r = 0 | 0.088 | 0.051 | 0.84 | 1.37 | 0.68 |
| 5  DMD, on-policy, r random | 0.093 | 0.662 | 0.51 | 0.48 | 1.93 |
| 6  MeanFlow, on-policy, r random | 0.080 | 0.037 | 0.95 | 0.76 | 0.67 |


In [8]:
d = np.load(demo_npz)
meta = json.loads(str(d["meta"]))
A.anim_samples(d, meta, 1, FPS, HOLD)
A.anim_samples(d, meta, 2, FPS, HOLD)
A.anim_directions(d, meta, FPS, HOLD)
A.f_pull_push_anchors()
show("samples_k1.gif", "directions.gif", "samples_k2.gif", "pull_push_anchors.png")

[anim] /Users/delcher/Desktop/self-info/contrastive-gen/materials/samples_k1.gif / .mp4 (73 frames)


[anim] /Users/delcher/Desktop/self-info/contrastive-gen/materials/samples_k2.gif / .mp4 (73 frames)


[anim] /Users/delcher/Desktop/self-info/contrastive-gen/materials/directions.gif / .mp4 (73 frames)


[figures] /Users/delcher/Desktop/self-info/contrastive-gen/materials/pull_push_anchors.png


![samples_k1.gif](../materials/samples_k1.gif)
![directions.gif](../materials/directions.gif)
![samples_k2.gif](../materials/samples_k2.gif)
![pull_push_anchors.png](../materials/pull_push_anchors.png)

**Ablations of the MeanFlow negative** (on-policy, $r = 0$, 2000 steps, seeds 0 and 1). `frozen` replaces
$\mathrm{sg}(u_\theta)$ in the negative by the initial student; `noJ` drops $J$. One-step SW2 to the teacher's
samples over training, next to the unablated student of part 4:

In [9]:
ABL = A.ART / "ablation" / "runs"
abl_common = ["--preset", PRESET, "--device", DEVICE, "--steps", "2000"]
jobs = [j for neg in ("frozen", "noJ")
        for j in jobs_for("mf-on-r0", ABL, seeds=(0, 1), extra=["--mf-neg", neg, "--tag", f"mf-{neg}"], common=abl_common)]
A.run_jobs(jobs, PARALLEL)
A.sw2_curves(ABL, ["mf-on-r0+mf-frozen", "mf-on-r0+mf-noJ"], steps=(0, 200, 500, 1000, 2000))
A.sw2_curves(DIST, ["mf-on-r0"], steps=(0, 200, 500, 1000, 2000))

[jobs] 0 already done, 4 to run


[mf-on-r0+mf-noJ s0] step   2000 sw2_k1 0.9060 sw2_k4 0.3409 coup 1.8690 | [mf-on-r0+mf-noJ s0] done in 16s


[mf-on-r0+mf-noJ s1] step   2000 sw2_k1 0.9060 sw2_k4 0.3409 coup 1.8690 | [mf-on-r0+mf-noJ s1] done in 16s


[mf-on-r0+mf-frozen s0] step   2000 sw2_k1 103.2687 sw2_k4 326.4231 coup 23594.1680 | [mf-on-r0+mf-frozen s0] done in 18s


[mf-on-r0+mf-frozen s1] step   2000 sw2_k1 112.1573 sw2_k4 470.1839 coup 27934.4824 | [mf-on-r0+mf-frozen s1] done in 18s


run                                  0       200       500      1000      2000
mf-on-r0+mf-frozen               0.906     3.841    69.015   111.125   107.713
mf-on-r0+mf-noJ                  0.906     0.906     0.906     0.906     0.906
run                                  0       200       500      1000      2000
mf-on-r0                         0.906     0.104     0.130     0.118     0.103


## 5. Positives and negatives from samples (Section 8)

Three variants, each with $N \in \{2, 4, 8, 16, 64, 256, 1024\}$ samples per step and three seeds:

* **native drift**: the drifting model of Deng et al. (2026), Algorithm 2, Laplacian kernel;
* **FM-ideal drift**: the same with a Gaussian kernel;
* **semi-drift**: DMD with the teacher's positive and the Gaussian-kernel posterior over the batch as negative.

In [10]:
VARIANTS = {
    "drift": lambda n: ["--variant", "drift-on-r0", "--drift-n", str(n), "--tag", f"n{n}"],
    "drift_gauss": lambda n: ["--variant", "drift-on-r0", "--drift-kernel", "gauss", "--drift-n", str(n), "--tag", f"g{n}"],
    "kneg": lambda n: ["--variant", "dmd-on-r0", "--dmd-neg", "kernel", "--drift-n", str(n), "--tag", f"kneg{n}"],
}
jobs = [[*VARIANTS[f](n), "--seed", str(s), "--root", str(A.default_root(f)), *COMMON]
        for n in sorted(A.NS, reverse=True) for f in A.FAMILIES for s in SEEDS]
A.run_jobs(jobs, PARALLEL)
kernel_npz = {f: A.export_kernel(f, A.default_root(f), PRESET, A.pick_device(DEVICE)) for f in A.FAMILIES}

[jobs] 0 already done, 63 to run


[drift-on-r0+n256 s2] step   3000 sw2_k1 0.0476 sw2_k4 0.0941 coup 0.2156 | [drift-on-r0+n256 s2] done in 44s


[drift-on-r0+n256 s0] step   3000 sw2_k1 0.0450 sw2_k4 0.0979 coup 0.2129 | [drift-on-r0+n256 s0] done in 44s


[drift-on-r0+n256 s1] step   3000 sw2_k1 0.0505 sw2_k4 0.1047 coup 0.2242 | [drift-on-r0+n256 s1] done in 45s


[drift-on-r0+g256 s2] step   3000 sw2_k1 0.0446 sw2_k4 0.1086 coup 1.7229 | [drift-on-r0+g256 s2] done in 48s


...

[kneg] N=16: 3 seeds


[kneg] N=64: 3 seeds


[kneg] N=256: 3 seeds


[kneg] N=1024: 3 seeds



kneg: final test KL vs held-out data (nats, kNN k=5; mean ± sd over seeds). Data floor: rev -0.002, fwd -0.004
     N       K=1 rev           fwd           sw2        K=2 rev           fwd           sw2        K=4 rev           fwd           sw2
     2   9.206±0.420    8.841±0.566  217.413±25.546   11.252±0.436   13.791±2.006  13297.394±6013.274   11.489±0.105   23.947±6.319  37314656.000±37897908.000
     4   5.020±0.398    4.357±0.318    4.310±0.639    5.792±0.842    5.154±0.774    7.273±3.615    6.146±1.100    5.084±1.176   17.698±17.378
     8   2.992±0.107    2.283±0.079    0.655±0.029    3.145±0.040    2.370±0.033    0.623±0.026    2.579±0.185    2.173±0.053    0.394±0.023
    16   1.862±0.064    1.584±0.072    0.259±0.002    1.892±0.025    1.625±0.009    0.254±0.006    1.519±0.048    1.315±0.131    0.176±0.005
    64   0.768±0.017    0.742±0.034    0.110±0.012    0.791±0.019    0.686±0.020    0.087±0.001    1.007±0.076    0.860±0.101    0.134±0.005
   256   0.389±0.025    0.411

The table of Section 8 (each cell: one-step KL(student‖data) / one-step SW2):

In [11]:
A.table_kernel()

| N | Native drift: Laplacian kernel | FM-ideal drift: Gaussian kernel | Semi-drift: teacher positive, Gaussian-kernel negative |
|---|---|---|---|
| 2 | 9.28 / 78.665 | 9.24 / 75.925 | 9.21 / 217.413 |
| 4 | 1.95 / 0.170 | 1.90 / 0.170 | 5.02 / 4.310 |
| 8 | 1.16 / 0.103 | 1.26 / 0.114 | 2.99 / 0.655 |
| 16 | 0.83 / 0.083 | 0.88 / 0.130 | 1.86 / 0.259 |
| 64 | 0.40 / 0.063 | 0.41 / 0.064 | 0.77 / 0.110 |
| 256 | 0.39 / 0.053 | 0.29 / 0.047 | 0.39 / 0.085 |
| 1024 | 0.28 / 0.052 | 0.25 / 0.041 | 0.51 / 0.059 |


In [12]:
ds = [np.load(kernel_npz[f]) for f in A.FAMILIES]
A.kl_vs_n_pair(ds)
A.anim_kernel_compare(ds, FPS, HOLD)
A.anim_drift_directions(ds[0], FPS, HOLD)
show("kl_vs_n_drift_drift_gauss_kneg.png", "kernel_samples.gif", "drift_directions.gif")

[kl] /Users/delcher/Desktop/self-info/contrastive-gen/materials/kl_vs_n_drift_drift_gauss_kneg.png


[anim] /Users/delcher/Desktop/self-info/contrastive-gen/materials/kernel_samples.gif / .mp4 (73 frames)


[anim] /Users/delcher/Desktop/self-info/contrastive-gen/materials/drift_directions.gif / .mp4 (73 frames)


![kl_vs_n_drift_drift_gauss_kneg.png](../materials/kl_vs_n_drift_drift_gauss_kneg.png)
![kernel_samples.gif](../materials/kernel_samples.gif)
![drift_directions.gif](../materials/drift_directions.gif)

## 6. Every asset the post links

In [13]:
post = (Path.cwd().parent / "contrastive_generation.md").read_text()
linked = sorted(set(re.findall(r"\]\((materials/[^)]+)\)", post)))
missing = [p for p in linked if not (Path.cwd().parent / p).exists()]
for p in linked:
    print("ok     " if p not in missing else "MISSING", p)
assert not missing, missing

ok      materials/directions.gif
ok      materials/drift_directions.gif
ok      materials/f2_tangent_vs_chord.png
ok      materials/kernel_samples.gif
ok      materials/kl_vs_n_drift_drift_gauss_kneg.png
ok      materials/pull_push_anchors.png
ok      materials/samples_k1.gif
ok      materials/samples_k2.gif
ok      materials/trajectory_distillation.gif
